In [1]:
import pandapower as pp
import pandapower.networks as nw
import math
import pandapower.plotting as plot
import matplotlib.pyplot as plt
import pandapower.plotting.plotly as pplotly
import pandapower.plotting as plot
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.colors as pcolors
import csv
import tqdm
from pathlib import Path
import json
from pandapower.networks.power_system_test_cases import case14, case24_ieee_rts

In [2]:
def plot_net(net, show=True):
    def ensure_bus_coords():
        if hasattr(net, "bus_geodata") and {"x", "y"}.issubset(net.bus_geodata.columns) and not net.bus_geodata.empty:
            coords = net.bus_geodata[["x", "y"]].astype(float)
            if "geo" not in net.bus.columns:
                net.bus["geo"] = pd.NA
            for bus, row in coords.iterrows():
                net.bus.at[bus, "geo"] = json.dumps({"type": "Point", "coordinates": [row.x, row.y]})
            return coords

        if "geo" not in net.bus.columns or net.bus.geo.dropna().empty:
            plot.create_generic_coordinates(net, overwrite=True)

        coords = net.bus.geo.dropna().apply(lambda g: pd.Series((json.loads(g) if isinstance(g, str) else g)["coordinates"], index=["x", "y"]))
        if coords.empty:
            raise ValueError("No bus coordinates are available for plotting.")
        return coords.astype(float)

    def col(df, name, default=np.nan):
        return df[name] if df is not None and name in df.columns else pd.Series(default, index=df.index if df is not None else pd.Index([]))

    def fmt(value, unit=""):
        return "N/A" if pd.isna(value) else f"{float(value):.2f}{unit}"

    def cost_map(et):
        cols = ["cp1_eur_per_mw", "cp2_eur_per_mw2"]
        if not hasattr(net, "poly_cost") or net.poly_cost.empty:
            return pd.DataFrame(columns=cols)
        keep = ["element"] + [c for c in cols if c in net.poly_cost.columns]
        cost = net.poly_cost.loc[net.poly_cost.et == et, keep]
        if cost.empty:
            return pd.DataFrame(columns=cols)
        return cost.drop_duplicates("element").set_index("element").reindex(columns=cols)

    def scatter_from_specs(specs, name, color, symbol, size):
        if not specs:
            return []
        return [go.Scatter(
            x=[s["x"] for s in specs],
            y=[s["y"] for s in specs],
            mode="markers",
            marker=dict(symbol=symbol, size=size, color=color, line=dict(color="black", width=0.8)),
            text=[s["hover"] for s in specs],
            hoverinfo="text",
            name=name,
            legendgroup=name,
        )]

    def add_arrows(elements, from_col, to_col, p_col, loading_col, coords, xs, ys, angles, colors):
        power = col(elements["res"], p_col).reindex(elements["table"].index)
        loading = col(elements["res"], loading_col).reindex(elements["table"].index)
        for idx, row in elements["table"].iterrows():
            fb, tb = row[from_col], row[to_col]
            if not row.in_service or fb not in coords.index or tb not in coords.index:
                continue
            p = power.at[idx]
            if pd.isna(p) or p == 0:
                continue
            x0, y0 = coords.loc[fb, ["x", "y"]]
            x1, y1 = coords.loc[tb, ["x", "y"]]
            if p < 0:
                x0, y0, x1, y1 = x1, y1, x0, y0
            xs.append((x0 + x1) / 2)
            ys.append((y0 + y1) / 2)
            angles.append(90 - np.degrees(np.arctan2(y1 - y0, x1 - x0)))
            colors.append("green" if pd.isna(loading.at[idx]) else pcolors.sample_colorscale("jet", [np.clip(loading.at[idx] / 100, 0, 1)])[0])

    pp.rundcopp(net)
    coords = ensure_bus_coords().reindex(net.bus.index).dropna()
    traces = []

    if not net.line.empty:
        p_from = col(net.res_line, "p_from_mw").reindex(net.line.index)
        loading = col(net.res_line, "loading_percent").reindex(net.line.index)
        cap = np.sqrt(3) * net.bus.vn_kv.reindex(net.line.from_bus).fillna(0).to_numpy() * col(net.line, "max_i_ka").fillna(np.nan).to_numpy()
        if "max_loading_percent" in net.line.columns:
            cap = cap * col(net.line, "max_loading_percent").reindex(net.line.index).fillna(100).to_numpy() / 100
        line_hover = pd.Series([
            "<br>".join([
                f"Line {idx}",
                f"From: Bus {row.from_bus}",
                f"To: Bus {row.to_bus}",
                f"P_flow: {fmt(abs(p_from.at[idx]), ' MW')}",
                f"Max Cap: {fmt(cap[i], ' MW')}",
                f"Loading: {fmt(loading.at[idx], ' %')}"
            ])
            for i, (idx, row) in enumerate(net.line.iterrows())
        ], index=net.line.index)
        line_traces = pplotly.create_line_trace(net, use_line_geo=False, cmap="jet", cmap_vals=loading.values, cmin=0, cmax=100, infofunc=line_hover, width=3)
        for trace in line_traces:
            if "marker" in trace and "showscale" in trace["marker"]:
                trace["marker"]["showscale"] = False
                trace["showlegend"] = False
        traces.extend(line_traces)

    if hasattr(net, "trafo") and not net.trafo.empty:
        trafo_p = col(net.res_trafo, "p_hv_mw").reindex(net.trafo.index)
        trafo_loading = col(net.res_trafo, "loading_percent").reindex(net.trafo.index)
        trafo_hover = pd.Series([
            "<br>".join([
                f"Transformer {idx}",
                f"HV Bus: {row.hv_bus}",
                f"LV Bus: {row.lv_bus}",
                f"P_flow: {fmt(abs(trafo_p.at[idx]), ' MW')}",
                f"Max Cap: {fmt(col(net.trafo, 'sn_mva').at[idx] * (col(net.trafo, 'max_loading_percent').at[idx] if 'max_loading_percent' in net.trafo.columns else 100) / 100, ' MW')}",
                f"Loading: {fmt(trafo_loading.at[idx], ' %')}"
            ])
            for idx, row in net.trafo.iterrows()
        ], index=net.trafo.index)
        traces.extend(pplotly.create_trafo_trace(net, trafos=net.trafo.index.tolist(), color="green", width=5, infofunc=trafo_hover, trace_name="Transformers"))

    lmp_col = next((c for c in ["lam_p", "marginal_price"] if hasattr(net, "res_bus") and c in net.res_bus.columns), None)
    bus_hover = []
    for bus in coords.index:
        parts = [f"Bus {bus}"]
        if "vn_kv" in net.bus.columns:
            parts.append(f"Vn: {fmt(net.bus.at[bus, 'vn_kv'], ' kV')}")
        if hasattr(net, "res_bus") and bus in net.res_bus.index:
            if "vm_pu" in net.res_bus.columns:
                parts.append(f"Vm: {fmt(net.res_bus.at[bus, 'vm_pu'], ' pu')}")
            if lmp_col:
                parts.append(f"LMP: {fmt(net.res_bus.at[bus, lmp_col], ' $/MWh')}")
        for label, table in [("Load", getattr(net, "load", None)), ("Gen", getattr(net, "gen", None)), ("Ext Grid", getattr(net, "ext_grid", None))]:
            if table is not None and not table.empty and "bus" in table.columns:
                ids = table.index[table.bus == bus].tolist()
                if ids:
                    parts.append(f"{label}: {ids}")
        bus_hover.append("<br>".join(parts))
    traces.append(go.Scatter(x=coords.x, y=coords.y, mode="markers", marker=dict(size=11, color="lightgray", line=dict(color="black", width=0.8)), text=bus_hover, hoverinfo="text", name="Buses"))

    specs = []
    component_defs = [
        ("gen", net.gen if hasattr(net, "gen") else None, net.res_gen if hasattr(net, "res_gen") else None, "Generators", "orange", "square", 14, cost_map("gen"), lambda i, r, res, cost: [f"Gen {i}", f"Bus: {r.bus}", f"P_out: {fmt(col(res, 'p_mw').at[i], ' MW')}", f"Min P: {fmt(col(net.gen, 'min_p_mw').at[i], ' MW')}", f"Max P: {fmt(col(net.gen, 'max_p_mw').at[i], ' MW')}", f"Cost: cp1={fmt(cost.at[i, 'cp1_eur_per_mw'] if i in cost.index else np.nan, ' $/MW')}, cp2={fmt(cost.at[i, 'cp2_eur_per_mw2'] if i in cost.index else np.nan, ' $/MW^2')}"]),
        ("load", net.load if hasattr(net, "load") else None, net.res_load if hasattr(net, "res_load") else None, "Loads", "blue", "diamond", 14, None, lambda i, r, res, cost: [f"Load {i}", f"Bus: {r.bus}", f"P_in: {fmt(col(res, 'p_mw').at[i], ' MW')}", f"LMP: {fmt(net.res_bus.at[r.bus, lmp_col], ' $/MWh') if lmp_col else 'N/A'}"]),
        ("ext_grid", net.ext_grid if hasattr(net, "ext_grid") else None, net.res_ext_grid if hasattr(net, "res_ext_grid") else None, "Ext Grid (Slack)", "red", "star", 16, cost_map("ext_grid"), lambda i, r, res, cost: [f"Ext Grid {i}", f"Bus: {r.bus}", f"P_out: {fmt(col(res, 'p_mw').at[i], ' MW')}", f"Min P: {fmt(col(net.ext_grid, 'min_p_mw').at[i], ' MW')}", f"Max P: {fmt(col(net.ext_grid, 'max_p_mw').at[i], ' MW')}", f"Cost: cp1={fmt(cost.at[i, 'cp1_eur_per_mw'] if i in cost.index else np.nan, ' $/MW')}, cp2={fmt(cost.at[i, 'cp2_eur_per_mw2'] if i in cost.index else np.nan, ' $/MW^2')}"]),
    ]

    for kind, table, res, name, color, symbol, size, costs, hover_fn in component_defs:
        if table is None or table.empty:
            continue
        for idx, row in table.iterrows():
            if row.bus not in coords.index:
                continue
            specs.append({"kind": kind, "bus": row.bus, "hover": "<br>".join(hover_fn(idx, row, res, costs)), "name": name, "color": color, "symbol": symbol, "size": size})

    if specs:
        radius = max((coords.x.max() - coords.x.min()), (coords.y.max() - coords.y.min())) * 0.04 or 0.5
        angle_pref = {"ext_grid": np.pi / 2, "gen": 0, "load": np.pi}
        connector_x, connector_y = [], []
        by_kind = {}
        for bus, group in pd.Series(specs).groupby([s["bus"] for s in specs]):
            items = sorted(group.tolist(), key=lambda s: (s["kind"], s["hover"]))
            for i, item in enumerate(items):
                angle = angle_pref.get(item["kind"], np.pi / 4) if len(items) == 1 else np.pi / 2 + 2 * np.pi * i / len(items)
                bx, by = coords.loc[bus, ["x", "y"]]
                item["x"] = bx + radius * np.cos(angle)
                item["y"] = by + radius * np.sin(angle)
                connector_x += [bx, item["x"], None]
                connector_y += [by, item["y"], None]
                by_kind.setdefault(item["name"], []).append(item)
        traces.append(go.Scatter(x=connector_x, y=connector_y, mode="lines", line=dict(color="rgba(80,80,80,0.5)", width=1), hoverinfo="skip", showlegend=False))
        for items in by_kind.values():
            first = items[0]
            traces.extend(scatter_from_specs(items, first["name"], first["color"], first["symbol"], first["size"]))

    arrow_x, arrow_y, arrow_angles, arrow_colors = [], [], [], []
    if not net.line.empty:
        add_arrows({"table": net.line, "res": net.res_line}, "from_bus", "to_bus", "p_from_mw", "loading_percent", coords, arrow_x, arrow_y, arrow_angles, arrow_colors)
    if hasattr(net, "trafo") and not net.trafo.empty:
        add_arrows({"table": net.trafo, "res": net.res_trafo}, "hv_bus", "lv_bus", "p_hv_mw", "loading_percent", coords, arrow_x, arrow_y, arrow_angles, arrow_colors)
    if arrow_x:
        traces.append(go.Scatter(x=arrow_x, y=arrow_y, mode="markers", marker=dict(symbol="triangle-up", size=14, color=arrow_colors, angle=arrow_angles, line=dict(color="black", width=0.5)), hoverinfo="skip", showlegend=False))

    fig = go.Figure(traces)
    dx = coords.x.max() - coords.x.min()
    dy = coords.y.max() - coords.y.min()
    mx = dx * 0.1 if dx else 1
    my = dy * 0.1 if dy else 1
    fig.update_layout(
        xaxis=dict(range=[coords.x.min() - mx, coords.x.max() + mx], title="X Coordinate"),
        yaxis=dict(range=[coords.y.min() - my, coords.y.max() + my], title="Y Coordinate", scaleanchor="x", scaleratio=1),
        width=1000,
        height=1000,
        showlegend=True,
        template="plotly_white",
        hovermode="closest",
    )
    if show:
        fig.show()
        return None
    return fig


In [3]:
net = case14()

In [4]:
net.trafo

,name,std_type,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,...,tap_min,tap_max,tap_step_percent,tap_step_degree,tap_pos,parallel,df,in_service,max_loading_percent,tap_changer_type
0,None,None,3,6,9900.0,135.0,14.000,2070.288,0.0,0.0,...,NaN,NaN,2.2,NaN,-1.0,1,1.0,True,100.0,Ratio
1,None,None,3,8,9900.0,135.0,0.208,5506.182,0.0,0.0,...,NaN,NaN,3.1,NaN,-1.0,1,1.0,True,100.0,Ratio
2,None,None,4,5,9900.0,135.0,0.208,2494.998,0.0,0.0,...,NaN,NaN,6.8,NaN,-1.0,1,1.0,True,100.0,Ratio
3,None,None,6,7,9900.0,14.0,12.000,1743.885,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,1,1.0,True,100.0,None
4,None,None,6,8,9900.0,14.0,0.208,1089.099,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,1,1.0,True,100.0,None


In [5]:
net.load

,name,bus,p_mw,q_mvar,const_z_p_percent,const_z_q_percent,const_i_p_percent,const_i_q_percent,sn_mva,scaling,in_service,type,controllable
0,None,1,21.7,12.7,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
1,None,2,94.2,19.0,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
2,None,3,47.8,-3.9,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
3,None,4,7.6,1.6,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
4,None,5,11.2,7.5,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
5,None,8,29.5,16.6,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
6,None,9,9.0,5.8,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
7,None,10,3.5,1.8,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
8,None,11,6.1,1.6,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
9,None,12,13.5,5.8,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False


In [6]:
net.gen

,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style
0,None,1,40.0,1.045,NaN,-40.0,50.0,1.0,False,True,0.0,None,True,140.0,0.0,<NA>,False,None
1,None,2,0.0,1.010,NaN,0.0,40.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
2,None,5,0.0,1.070,NaN,-6.0,24.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
3,None,7,0.0,1.090,NaN,-6.0,24.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None


In [7]:
net.poly_cost

,element,et,cp0_eur,cp1_eur_per_mw,cp2_eur_per_mw2,cq0_eur,cq1_eur_per_mvar,cq2_eur_per_mvar2
0,0,ext_grid,0.0,20.0,0.043029,0.0,0.0,0.0
1,0,gen,0.0,20.0,0.250000,0.0,0.0,0.0
2,1,gen,0.0,40.0,0.010000,0.0,0.0,0.0
3,2,gen,0.0,40.0,0.010000,0.0,0.0,0.0
4,3,gen,0.0,40.0,0.010000,0.0,0.0,0.0


In [8]:
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,max_loading_percent,geo
0,None,None,0,1,1.0,3.532005,10.783732,768.484773,0.0,42.339020,1.0,1,ol,True,100.0,None
1,None,None,0,4,1.0,9.846967,40.649040,716.088084,0.0,42.339020,1.0,1,ol,True,100.0,None
2,None,None,1,2,1.0,8.563928,36.080033,637.493051,0.0,42.339020,1.0,1,ol,True,100.0,None
3,None,None,1,3,1.0,10.590547,32.134320,494.857619,0.0,42.339020,1.0,1,ol,True,100.0,None
4,None,None,1,4,1.0,10.379138,31.689630,503.590401,0.0,42.339020,1.0,1,ol,True,100.0,None
5,None,None,2,3,1.0,12.212573,31.170217,186.299339,0.0,42.339020,1.0,1,ol,True,100.0,None
6,None,None,3,4,1.0,2.433038,7.674548,0.000000,0.0,42.339020,1.0,1,ol,True,100.0,None
7,None,None,5,10,1.0,0.000041,0.000086,0.000000,0.0,27479.652235,1.0,1,ol,True,100.0,None
8,None,None,5,11,1.0,0.000053,0.000111,0.000000,0.0,27479.652235,1.0,1,ol,True,100.0,None
9,None,None,5,12,1.0,0.000029,0.000056,0.000000,0.0,27479.652235,1.0,1,ol,True,100.0,None


In [9]:

net.line['max_loading_percent'] = 100
net.line['max_i_ka'] = 100 / (np.sqrt(3) * net.bus.loc[net.line.from_bus, 'vn_kv'].to_numpy())
net.trafo['sn_mva'] = 100

In [10]:
plot_net(net)

gen vm_pu > bus max_vm_pu for gens [2 3]. Setting bus limit for these gens.


In [11]:
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar,lam_p,lam_q
0,1.0,0.000000,-154.645507,NaN,33.308576,0.0
1,1.0,-3.390191,-22.398347,NaN,42.049173,0.0
2,1.0,-7.945343,39.041442,NaN,41.103171,0.0
3,1.0,-8.054836,47.800000,NaN,40.285902,0.0
4,1.0,-6.983286,7.600000,NaN,39.664168,0.0
5,1.0,-522.577339,11.197556,NaN,39.998810,0.0
6,1.0,-322.055912,0.000000,NaN,40.101903,0.0
7,1.0,-271.146623,-5.095145,NaN,40.101903,0.0
8,1.0,-522.749712,29.500000,NaN,40.002931,0.0
9,1.0,-523.148260,9.000000,NaN,40.002198,0.0


In [12]:
net.load.loc[8, "p_mw"] += 1

In [13]:
plot_net(net)

gen vm_pu > bus max_vm_pu for gens [2 3]. Setting bus limit for these gens.


In [14]:
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar,lam_p,lam_q
0,1.0,0.000000,-154.709650,NaN,33.314096,0.0
1,1.0,-3.390191,-22.411952,NaN,42.055976,0.0
2,1.0,-7.930971,38.708246,NaN,41.109835,0.0
3,1.0,-8.060699,47.800000,NaN,40.292446,0.0
4,1.0,-6.991483,7.600000,NaN,39.670621,0.0
5,1.0,-525.825351,10.934352,NaN,40.005312,0.0
6,1.0,-322.803570,0.000000,NaN,40.108420,0.0
7,1.0,-268.638475,-5.420995,NaN,40.108420,0.0
8,1.0,-525.929709,29.500000,NaN,40.009433,0.0
9,1.0,-526.340345,9.000000,NaN,40.008701,0.0
